In [10]:
import pandas as pd
import re
from pathlib import Path

# =========================
# 1. READ LOG FILE
# =========================

DATA_PATH = Path("../data/ssh_auth_sample.log")

with open(DATA_PATH, "r", encoding="utf-8") as file:
    logs = file.readlines()


# =========================
# 2. PARSE SSH LOGS
# =========================

def parse_ssh_log(line):
    pattern = r"(?P<month>\w+)\s+(?P<day>\d+)\s+(?P<time>\d+:\d+:\d+)\s+(?P<host>\w+)\s+sshd\[(?P<pid>\d+)\]:\s+(?P<message>.*)"
    match = re.match(pattern, line)

    if not match:
        return None

    data = match.groupdict()
    message = data["message"]

    ip_match = re.search(r"from\s+(\d+\.\d+\.\d+\.\d+)", message)
    port_match = re.search(r"port\s+(\d+)", message)
    user_match = re.search(r"for\s+(?:invalid user\s+)?(\w+)", message)

    if "Failed password" in message:
        event_type = "failed_login"
    elif "Accepted password" in message:
        event_type = "successful_login"
    else:
        event_type = "other"

    return {
        "month": data["month"],
        "day": int(data["day"]),
        "time": data["time"],
        "host": data["host"],
        "pid": data["pid"],
        "event_type": event_type,
        "username": user_match.group(1) if user_match else None,
        "source_ip": ip_match.group(1) if ip_match else None,
        "source_port": port_match.group(1) if port_match else None,
        "raw_log": line.strip()
    }


parsed_logs = []

for line in logs:
    parsed = parse_ssh_log(line)
    if parsed is not None:
        parsed_logs.append(parsed)

df = pd.DataFrame(parsed_logs)

df["timestamp"] = pd.to_datetime(
    df["month"] + " " + df["day"].astype(str) + " 2026 " + df["time"],
    format="%b %d %Y %H:%M:%S"
)

df = df.sort_values("timestamp").reset_index(drop=True)


# =========================
# 3. DETECTION RULES
# =========================

def detect_ssh_bruteforce(df, threshold=5, time_window="5min"):
    alerts = []

    failed_logins = df[df["event_type"] == "failed_login"].copy()
    failed_logins = failed_logins.sort_values("timestamp")

    for source_ip, group in failed_logins.groupby("source_ip"):
        group = group.set_index("timestamp")
        rolling_counts = group["event_type"].rolling(time_window).count()

        if rolling_counts.max() >= threshold:
            alerts.append({
                "alert_type": "SSH Brute Force",
                "severity": "High",
                "source_ip": source_ip,
                "username": None,
                "attempts": int(len(group)),
                "first_seen": group.index.min(),
                "last_seen": group.index.max(),
                "targeted_users": group["username"].dropna().unique().tolist(),
                "description": f"Possible SSH brute-force attack from {source_ip} with {len(group)} failed login attempts.",
                "recommendation": "Block the source IP, check for successful logins, and inspect authentication logs."
            })

    return alerts


def detect_success_after_failures(df, failure_threshold=3, time_window_minutes=10):
    alerts = []

    df_sorted = df.sort_values("timestamp")

    for source_ip, group in df_sorted.groupby("source_ip"):
        group = group.sort_values("timestamp")

        successful_logins = group[group["event_type"] == "successful_login"]

        for _, success_row in successful_logins.iterrows():
            success_time = success_row["timestamp"]
            window_start = success_time - pd.Timedelta(minutes=time_window_minutes)

            previous_failures = group[
                (group["event_type"] == "failed_login") &
                (group["timestamp"] >= window_start) &
                (group["timestamp"] < success_time)
            ]

            if len(previous_failures) >= failure_threshold:
                alerts.append({
                    "alert_type": "Successful Login After Multiple Failures",
                    "severity": "Critical",
                    "source_ip": source_ip,
                    "username": success_row["username"],
                    "attempts": int(len(previous_failures)),
                    "first_seen": previous_failures["timestamp"].min(),
                    "last_seen": success_time,
                    "targeted_users": previous_failures["username"].dropna().unique().tolist(),
                    "description": f"Successful SSH login for user '{success_row['username']}' after {len(previous_failures)} failed attempts from {source_ip}.",
                    "recommendation": "Immediately verify whether the login was legitimate. Check session activity, commands executed, and consider resetting credentials."
                })

    return alerts


# =========================
# 4. RISK SCORING
# =========================

def calculate_risk_score(alert):
    score = 0

    if alert["severity"] == "Low":
        score += 20
    elif alert["severity"] == "Medium":
        score += 45
    elif alert["severity"] == "High":
        score += 70
    elif alert["severity"] == "Critical":
        score += 90

    if alert["username"] == "root":
        score += 10

    if "root" in alert["targeted_users"]:
        score += 10

    return min(score, 100)


# =========================
# 5. RUN DETECTIONS
# =========================

alerts = []

alerts.extend(detect_ssh_bruteforce(df))
alerts.extend(detect_success_after_failures(df))

clean_alerts = pd.DataFrame(alerts)

clean_alerts["risk_score"] = clean_alerts.apply(calculate_risk_score, axis=1)

clean_alerts = clean_alerts[
    [
        "alert_type",
        "severity",
        "risk_score",
        "source_ip",
        "username",
        "attempts",
        "first_seen",
        "last_seen",
        "targeted_users",
        "description",
        "recommendation"
    ]
]

clean_alerts

,alert_type,severity,risk_score,source_ip,username,attempts,first_seen,last_seen,targeted_users,description,recommendation
0,SSH Brute Force,High,80,185.23.45.10,None,6,2026-01-10 10:01:12,2026-01-10 10:03:55,"[root, admin, test]",Possible SSH brute-force attack from 185.23.45...,"Block the source IP, check for successful logi..."
1,Successful Login After Multiple Failures,Critical,100,185.23.45.10,root,6,2026-01-10 10:01:12,2026-01-10 10:04:20,"[root, admin, test]",Successful SSH login for user 'root' after 6 f...,Immediately verify whether the login was legit...


In [11]:
OUTPUT_PATH = Path("../outputs/ssh_alerts.csv")

clean_alerts.to_csv(OUTPUT_PATH, index=False)

print(f"Alerts exported to: {OUTPUT_PATH}")

Alerts exported to: ..\outputs\ssh_alerts.csv


In [12]:
REPORT_PATH = Path("../outputs/ssh_soc_report.txt")

def generate_soc_report(clean_alerts):
    total_alerts = len(clean_alerts)
    critical_alerts = len(clean_alerts[clean_alerts["severity"] == "Critical"])
    high_alerts = len(clean_alerts[clean_alerts["severity"] == "High"])
    unique_ips = clean_alerts["source_ip"].nunique()

    report = []
    report.append("AI SOC Log Analyzer - SSH Security Report")
    report.append("=" * 55)
    report.append("")
    report.append("Summary")
    report.append("-" * 20)
    report.append(f"Total alerts detected: {total_alerts}")
    report.append(f"Critical alerts: {critical_alerts}")
    report.append(f"High alerts: {high_alerts}")
    report.append(f"Unique suspicious IPs: {unique_ips}")
    report.append("")

    report.append("Detected Alerts")
    report.append("-" * 20)

    for index, row in clean_alerts.iterrows():
        report.append(f"Alert #{index + 1}")
        report.append(f"Type: {row['alert_type']}")
        report.append(f"Severity: {row['severity']}")
        report.append(f"Risk score: {row['risk_score']}/100")
        report.append(f"Source IP: {row['source_ip']}")
        report.append(f"Username: {row['username']}")
        report.append(f"Attempts: {row['attempts']}")
        report.append(f"First seen: {row['first_seen']}")
        report.append(f"Last seen: {row['last_seen']}")
        report.append(f"Targeted users: {row['targeted_users']}")
        report.append(f"Description: {row['description']}")
        report.append(f"Recommendation: {row['recommendation']}")
        report.append("")

    report.append("SOC Analyst Notes")
    report.append("-" * 20)
    report.append(
        "The activity detected suggests a possible SSH brute-force attack, "
        "followed by a successful login. This pattern should be treated as highly suspicious, "
        "especially because the root account was targeted."
    )
    report.append("")
    report.append("Recommended next steps:")
    report.append("1. Verify whether the successful root login was legitimate.")
    report.append("2. Review shell history and commands executed during the session.")
    report.append("3. Check whether the source IP appears in other systems or firewall logs.")
    report.append("4. Temporarily block the source IP if the activity is unauthorized.")
    report.append("5. Enforce SSH hardening: disable root login, use key-based authentication, and enable rate limiting.")

    return "\n".join(report)

report_text = generate_soc_report(clean_alerts)

with open(REPORT_PATH, "w", encoding="utf-8") as file:
    file.write(report_text)

print(report_text)
print(f"\nReport exported to: {REPORT_PATH}")

AI SOC Log Analyzer - SSH Security Report

Summary
--------------------
Total alerts detected: 2
Critical alerts: 1
High alerts: 1
Unique suspicious IPs: 1

Detected Alerts
--------------------
Alert #1
Type: SSH Brute Force
Severity: High
Risk score: 80/100
Source IP: 185.23.45.10
Username: None
Attempts: 6
First seen: 2026-01-10 10:01:12
Last seen: 2026-01-10 10:03:55
Targeted users: ['root', 'admin', 'test']
Description: Possible SSH brute-force attack from 185.23.45.10 with 6 failed login attempts.
Recommendation: Block the source IP, check for successful logins, and inspect authentication logs.

Alert #2
Type: Successful Login After Multiple Failures
Severity: Critical
Risk score: 100/100
Source IP: 185.23.45.10
Username: root
Attempts: 6
First seen: 2026-01-10 10:01:12
Last seen: 2026-01-10 10:04:20
Targeted users: ['root', 'admin', 'test']
Description: Successful SSH login for user 'root' after 6 failed attempts from 185.23.45.10.
Recommendation: Immediately verify whether the l

In [13]:
def prepare_ai_context(clean_alerts):
    context = []
    
    context.append("Security alerts detected from SSH authentication logs.")
    context.append("")
    
    for index, row in clean_alerts.iterrows():
        context.append(f"Alert {index + 1}:")
        context.append(f"- Alert type: {row['alert_type']}")
        context.append(f"- Severity: {row['severity']}")
        context.append(f"- Risk score: {row['risk_score']}/100")
        context.append(f"- Source IP: {row['source_ip']}")
        context.append(f"- Username: {row['username']}")
        context.append(f"- Attempts: {row['attempts']}")
        context.append(f"- First seen: {row['first_seen']}")
        context.append(f"- Last seen: {row['last_seen']}")
        context.append(f"- Targeted users: {row['targeted_users']}")
        context.append(f"- Description: {row['description']}")
        context.append("")
    
    return "\n".join(context)

ai_context = prepare_ai_context(clean_alerts)

print(ai_context)

Security alerts detected from SSH authentication logs.

Alert 1:
- Alert type: SSH Brute Force
- Severity: High
- Risk score: 80/100
- Source IP: 185.23.45.10
- Username: None
- Attempts: 6
- First seen: 2026-01-10 10:01:12
- Last seen: 2026-01-10 10:03:55
- Targeted users: ['root', 'admin', 'test']
- Description: Possible SSH brute-force attack from 185.23.45.10 with 6 failed login attempts.

Alert 2:
- Alert type: Successful Login After Multiple Failures
- Severity: Critical
- Risk score: 100/100
- Source IP: 185.23.45.10
- Username: root
- Attempts: 6
- First seen: 2026-01-10 10:01:12
- Last seen: 2026-01-10 10:04:20
- Targeted users: ['root', 'admin', 'test']
- Description: Successful SSH login for user 'root' after 6 failed attempts from 185.23.45.10.



In [28]:
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai

CONFIG_PATH = Path("../config.env")

load_dotenv(CONFIG_PATH, override=True)

api_key = os.getenv("GEMINI_API_KEY")

if api_key is None:
    raise ValueError("GEMINI_API_KEY not found. Check your config.env file.")

client = genai.Client(api_key=api_key)

prompt = f"""
You are a SOC Level 2 cybersecurity analyst.

Analyze the following SSH security alerts.

Return a clear and practical SOC investigation report with:

1. Executive summary
2. Technical analysis
3. Likely attack scenario
4. Indicators of compromise
5. Investigation steps
6. Recommended remediation

Important rules:
- Do not claim confirmed compromise unless the evidence explicitly proves it.
- Use "suspected unauthorized access" or "potential compromise" when the evidence is suspicious but not fully confirmed.
- Keep the report concise, professional, and useful for a SOC analyst.
- Focus on practical investigation and remediation steps.

Security context:
{ai_context}
"""

response = client.models.generate_content(
    model="gemini-flash-lite-latest",
    contents=prompt
)

ai_report = response.text

print(ai_report)

**SOC Investigation Report: SSH Authentication Incident**

**Date:** 2026-01-10
**Incident ID:** INC-20260110-001
**Analyst:** SOC Level 2
**Status:** Under Investigation / Action Required

---

### 1. Executive Summary
On 2026-01-10, at 10:01:12, the security monitoring system detected a series of SSH authentication events originating from source IP `185.23.45.10`. The activity transitioned from a brute-force pattern targeting common administrative usernames (`root`, `admin`, `test`) to a **successful authentication** for the `root` account at 10:04:20. Given the successful access to a highly privileged account following automated brute-force attempts, this event is treated as a **potential compromise**.

### 2. Technical Analysis
*   **Source IP:** `185.23.45.10` (Geolocation and reputation check required)
*   **Attack Vector:** SSH Password/Credential Stuffing
*   **Target:** `root` account on the host server
*   **Timeline:**
    *   10:01:12: Initial brute-force attempts begin.
  

In [26]:
AI_REPORT_PATH = Path("../outputs/ssh_ai_soc_report.txt")

with open(AI_REPORT_PATH, "w", encoding="utf-8") as file:
    file.write(ai_report)

print(f"AI SOC report exported to: {AI_REPORT_PATH}")

AI SOC report exported to: ..\outputs\ssh_ai_soc_report.txt
